### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import sys
sys.path.append('./utils')
sys.path.append('/home/vino/.cache/huggingface/hub')
from svg_processor import SVGSanitizer, SVGProcessor,svg_constraints
from siglip_class import SVGMetricEvaluator

### Random seed for reproducibility

In [3]:
import torch
import random
import numpy as np
import multiprocessing as mp

mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [4]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc


class Model:
    
    def __init__(self):

        self.model_path="./lora/Llama_32_3B_Instruct_lora_fp16_r256_s2000_i1000_msl2048"
        self.model = LLM(
            model=self.model_path,
            max_model_len=1024,
            gpu_memory_utilization=0.85,
            dtype="half",
            seed=123,
            disable_log_stats=True
        )

       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     


    def _format_prompt(self, description: str) -> str:
        return  f"""Below is an instruction that describes a task, paired with an input that provides further context. 
                Write a response that appropriately completes the request.
                
                ### Instruction:
                Generate a SVG code for the given input:
                
                ### Input:
                {description}
                
                ### Response:
                """
    
    def get_response(self, descriptions):
        
        formatted_input = [self._format_prompt(desc) for desc in descriptions]
        sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=1024,n=1)
        outputs = self.model.generate(formatted_input, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
        return output_list
    
    def predict(self, descriptions: list[str], max_new_tokens=1024) -> list[str]:
        output_decoded_list = self.get_response(descriptions)
        final_svg_code_list = []
    
        for description, output in zip(descriptions, output_decoded_list):
            base_svg = SVGProcessor.clean_and_extract_svgs(output, self.default_svg)
            clean_svg = self.sanitizer.enforce_constraints(base_svg)
            final_svg = SVGProcessor.svg_conversion_check(description, clean_svg, self.default_svg)
            final_svg_code_list.append(final_svg)
    
        return final_svg_code_list


INFO 04-21 20:26:41 [__init__.py:239] Automatically detected platform cuda.


In [5]:
model=Model()

WARNING 04-21 20:26:42 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-21 20:26:46 [config.py:585] This model supports multiple tasks: {'generate', 'score', 'classify', 'reward', 'embed'}. Defaulting to 'generate'.
INFO 04-21 20:26:47 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-21 20:26:48 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/Llama_32_3B_Instruct_lora_fp16_r256_s2000_i1000_msl2048', speculative_config=None, tokenizer='./lora/Llama_32_3B_Instruct_lora_fp16_r256_s2000_i1000_msl2048', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=De

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-21 20:26:54 [loader.py:447] Loading weights took 5.50 seconds
INFO 04-21 20:26:54 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 5.652651 seconds
INFO 04-21 20:27:00 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/a118eb1931/rank_0_0 for vLLM's torch.compile
INFO 04-21 20:27:00 [backends.py:425] Dynamo bytecode transform time: 5.76 s


[rank0]:W0421 20:27:01.385000 16369 site-packages/torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode


INFO 04-21 20:27:02 [backends.py:132] Cache the graph of shape None for later use
INFO 04-21 20:27:18 [backends.py:144] Compiling a graph for general shape takes 17.94 s
INFO 04-21 20:27:29 [monitor.py:33] torch.compile takes 23.70 s in total
INFO 04-21 20:27:29 [kv_cache_utils.py:566] GPU KV cache size: 18,432 tokens
INFO 04-21 20:27:29 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 18.00x
INFO 04-21 20:27:46 [gpu_model_runner.py:1534] Graph capturing finished in 16 secs, took 0.38 GiB
INFO 04-21 20:27:46 [core.py:151] init engine (profile, create kv cache, warmup model) took 51.41 seconds


In [6]:
#tmp=model.predict(['sun rising in the east','A golden goose with a fish'])

In [7]:
#print(tmp[0])

In [8]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test_vqa.csv',header=[0])
print(df.shape)
df.head(2)

(75, 7)


,description,gpt_svg,gpt_score_sl,response,vqa_pair,response_2,gpt_svg_2
0,"'Vibrant autumn forest',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117,Here is the visual question answering (VQA) pa...,"{'description': 'Vibrant autumn forest', 'ques...","Here's an improved SVG representation of a ""Vi...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,"'Morning dew on grass',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.987024,Here is a visual question answering (VQA) pair...,"{'description': 'Morning dew on grass', 'quest...","Here's an improved SVG representation of ""Morn...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [9]:
# from tqdm import tqdm
# tqdm.pandas()
# df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [10]:
from tqdm import tqdm
description_list = [s.strip(" ',") for s in df['description'].to_list()]
batch_size = 12
results = []

for i in tqdm(range(0, len(description_list), batch_size), desc="Batch prediction"):
    batch = description_list[i:i + batch_size]
    batch_result = model.predict(batch)  # Ensure this handles a list of inputs
    results.extend(batch_result)


Batch prediction:   0%|                                   | 0/7 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/12 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   8%| | 1/12 [00:03<00:40,  3.65s/it, est. speed input: 16.70
cessed prompts:  17%|▏| 2/12 [00:06<00:28,  2.89s/it, est. speed input: 20.14
cessed prompts:  25%|▎| 3/12 [00:07<00:18,  2.06s/it, est. speed input: 25.68
cessed prompts:  33%|▎| 4/12 [00:08<00:15,  1.88s/it, est. speed input: 27.84
cessed prompts:  42%|▍| 5/12 [00:09<00:09,  1.37s/it, est. speed input: 33.36
cessed prompts:  50%|▌| 6/12 [00:10<00:07,  1.27s/it, est. speed input: 35.74
cessed prompts:  58%|▌| 7/12 [00:10<00:05,  1.03s/it, est. speed input: 39.57
cessed prompts:  67%|▋| 8/12 [00:11<00:04,  1.04s/it, est. speed input: 41.15
Processed prompts: 100%|█| 12/12 [00:13<00:00,  1.08s/it, est. speed input: 56.3
ERROR:root:SVG Parse Error: error parsing attribute name, line 1, column 2402 (<string>, line 1). Returning default SVG.
ERROR:root:SVG 

In [11]:
df['svg_3']=results

In [12]:
model.close_model()

In [13]:
#SigLip Score
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

100%|███████████████████████████████████████████| 75/75 [00:04<00:00, 15.03it/s]


In [14]:
df['svg_score_3'].mean()

0.6152778304804085